# NB03 Homework Solutions

Answers to the "Homework / Practice Ideas" section of
`NB03_NumPy_Array_Mechanics_Dtypes_Indexing_and_Views.ipynb`:

1. Load `ship_fuel_efficiency.csv` as a NumPy array (not a DataFrame) and boolean-mask rows where `ship_type == "Tanker Ship"`.
2. Demonstrate the view/copy distinction with `np.shares_memory`.
3. Use `np.meshgrid` to build a 5x5 grid spanning -2 to 2 and compute `z = x**2 + y**2`.
4. Compare the real memory footprint of the Sonar dataset as float64/float32/float16, and check where `np.allclose` starts reporting differences.
5. Fancy-index a shuffled copy of the Sonar dataset's row order and confirm with `np.shares_memory` it is a genuine copy.


## 1. `ship_fuel_efficiency.csv` as a NumPy array, boolean-masked by `ship_type`

The class notebook loads the Sonar dataset as a plain NumPy array with
`urllib.request` + manual line splitting (Section 2). I follow the same
pattern with `ship_fuel_efficiency.csv`, using Python's `csv` module to parse
rows (the class notebook itself uses `csv.DictReader` for this exact dataset in
`NB05`), then converting to a NumPy array of `dtype=object` since the columns
are mixed string/numeric -- a plain NumPy array does not have per-column dtypes
the way a DataFrame does.


In [1]:
import numpy as np
import csv

import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv", "ship_fuel_efficiency.csv")

with open("ship_fuel_efficiency.csv") as f:
    reader = csv.reader(f)
    header = next(reader)
    fuel_rows = list(reader)

fuel_array = np.array(fuel_rows, dtype=object)
print("Header:", header)
print("Shape:", fuel_array.shape)

ship_type_col = header.index("ship_type")
is_tanker = fuel_array[:, ship_type_col] == "Tanker Ship"
tanker_rows = fuel_array[is_tanker]

print(f"\nTanker Ship rows: {is_tanker.sum()} / {fuel_array.shape[0]}")
tanker_rows[:5]


Header: ['ship_id', 'ship_type', 'route_id', 'month', 'distance', 'fuel_type', 'fuel_consumption', 'CO2_emissions', 'weather_conditions', 'engine_efficiency']
Shape: (1440, 10)

Tanker Ship rows: 408 / 1440


array([['NG005', 'Tanker Ship', 'Escravos-Lagos', 'January', '336.53',
        'Diesel', '15596.39', '42180.94', 'Calm', '72.59'],
       ['NG005', 'Tanker Ship', 'Escravos-Lagos', 'February', '494.24',
        'HFO', '18375.75', '46753.73', 'Stormy', '94.18'],
       ['NG005', 'Tanker Ship', 'Warri-Bonny', 'March', '159.49', 'HFO',
        '5906.8', '17358.94', 'Calm', '74.34'],
       ['NG005', 'Tanker Ship', 'Port Harcourt-Lagos', 'April', '363.22',
        'Diesel', '17778.61', '53142.22', 'Calm', '75.92'],
       ['NG005', 'Tanker Ship', 'Warri-Bonny', 'May', '77.01', 'HFO',
        '2362.84', '6230.95', 'Moderate', '71.15']], dtype=object)

## 2. View vs. copy: `np.shares_memory`

Following Section 5's pattern exactly (basic slicing returns a view, `.copy()`
or fancy indexing returns a genuine copy), I slice out a 2-column subset of the
Sonar dataset, modify it, and check whether the original array changed.


In [2]:
import urllib.request
sonar_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data"
raw_lines = urllib.request.urlopen(sonar_url).read().decode("utf-8").strip().split("\n")
sonar_features = np.array([[float(x) for x in line.split(",")[:-1]] for line in raw_lines])
sonar_labels = np.array([line.split(",")[-1] for line in raw_lines])

original_first_values = sonar_features[0, :2].copy()  # protect the real data before this demo modifies it

subset_view = sonar_features[:, :2]   # basic slicing -> view
print("Basic-slice subset shares memory with original:", np.shares_memory(subset_view, sonar_features))

subset_view[0, 0] = -999.0
print("After modifying the subset, original[0,0] =", sonar_features[0, 0], "(was", original_first_values[0], ")")

sonar_features[0, 0] = original_first_values[0]  # restore the real data

subset_copy = sonar_features[:, [0, 1]]   # fancy indexing -> copy
print("\nFancy-indexed subset shares memory with original:", np.shares_memory(subset_copy, sonar_features))
subset_copy[0, 0] = -999.0
print("After modifying the fancy-indexed copy, original[0,0] =", sonar_features[0, 0])


Basic-slice subset shares memory with original: True
After modifying the subset, original[0,0] = -999.0 (was 0.02 )

Fancy-indexed subset shares memory with original: False
After modifying the fancy-indexed copy, original[0,0] = 0.02


Basic slicing (`sonar_features[:, :2]`) shares memory with the original array,
so modifying the slice modified the original in place -- exactly the "real,
common, silent" bug the class notebook warns about. Fancy indexing
(`sonar_features[:, [0, 1]]`) returned a genuine copy: `np.shares_memory`
reported `False`, and modifying it left the original untouched.


## 3. `np.meshgrid`: 5x5 grid from -2 to 2, `z = x**2 + y**2`


In [3]:
coords_1d = np.linspace(-2, 2, 5)
x_grid, y_grid = np.meshgrid(coords_1d, coords_1d)

z_grid = x_grid**2 + y_grid**2

print("x_grid:\n", x_grid)
print("\ny_grid:\n", y_grid)
print("\nz = x**2 + y**2:\n", z_grid)


x_grid:
 [[-2. -1.  0.  1.  2.]
 [-2. -1.  0.  1.  2.]
 [-2. -1.  0.  1.  2.]
 [-2. -1.  0.  1.  2.]
 [-2. -1.  0.  1.  2.]]

y_grid:
 [[-2. -2. -2. -2. -2.]
 [-1. -1. -1. -1. -1.]
 [ 0.  0.  0.  0.  0.]
 [ 1.  1.  1.  1.  1.]
 [ 2.  2.  2.  2.  2.]]

z = x**2 + y**2:
 [[8. 5. 4. 5. 8.]
 [5. 2. 1. 2. 5.]
 [4. 1. 0. 1. 4.]
 [5. 2. 1. 2. 5.]
 [8. 5. 4. 5. 8.]]


The grid is symmetric around the origin as expected for `x**2 + y**2`: the
minimum (0.0) sits exactly at the center cell `(x=0, y=0)`, and the four
corners (`x, y = +/-2`) all reach the maximum value of 8.0, since `z` only
depends on distance from the origin.


## 4. Sonar dataset memory footprint: float64 vs. float32 vs. float16

Extending Section 2's `float64` vs. `float32` comparison to also include
`float16`, and checking where `np.allclose` starts reporting real numerical
differences against the original `float64` array.


In [4]:
sonar_f32 = sonar_features.astype(np.float32)
sonar_f16 = sonar_features.astype(np.float16)

print(f"float64: {sonar_features.nbytes / 1024:.1f} KB")
print(f"float32: {sonar_f32.nbytes / 1024:.1f} KB  ({sonar_features.nbytes / sonar_f32.nbytes:.1f}x smaller)")
print(f"float16: {sonar_f16.nbytes / 1024:.1f} KB  ({sonar_features.nbytes / sonar_f16.nbytes:.1f}x smaller)")

for name, arr, tol_rtol in [("float32", sonar_f32, 1e-5), ("float16", sonar_f16, 1e-3)]:
    close_default = np.allclose(sonar_features, arr.astype(np.float64))
    close_loose = np.allclose(sonar_features, arr.astype(np.float64), rtol=tol_rtol)
    max_abs_diff = np.abs(sonar_features - arr.astype(np.float64)).max()
    print(f"\n{name}: np.allclose (default tol) = {close_default}, "
          f"np.allclose (rtol={tol_rtol}) = {close_loose}, max abs diff = {max_abs_diff:.6f}")


float64: 97.5 KB
float32: 48.8 KB  (2.0x smaller)
float16: 24.4 KB  (4.0x smaller)

float32: np.allclose (default tol) = True, np.allclose (rtol=1e-05) = True, max abs diff = 0.000000

float16: np.allclose (default tol) = False, np.allclose (rtol=0.001) = True, max abs diff = 0.000244


**Where does `np.allclose` start reporting real differences?** With
`np.allclose`'s default tolerance (`rtol=1e-5, atol=1e-8`), `float32` still
passes (the rounding error is far below default tolerance) but `float16`
already fails -- the actual max absolute difference for `float16` is printed
above and is well above `float16`'s much coarser precision (about 3 decimal
digits, vs. float64's ~15-17). Since the Sonar dataset's real values are all in
the 0.0-1.0 range, `float16`'s roughly `1e-3`-scale rounding error is
proportionally large enough to fail the default tolerance immediately, and only
passes once the tolerance is loosened to match `float16`'s own precision. This
is the concrete version of the class notebook's abstract warning: `dtype`
choice is a real memory/precision trade-off, not just an implementation
detail.


## 5. Fancy-indexing shuffle of the Sonar dataset's row order


In [5]:
rng = np.random.default_rng(42)
shuffled_order = rng.permutation(sonar_features.shape[0])

shuffled_sonar = sonar_features[shuffled_order]

print("Shuffled array shares memory with original:", np.shares_memory(shuffled_sonar, sonar_features))
print("Same shape:", shuffled_sonar.shape == sonar_features.shape)
print("Row order actually changed:", not np.array_equal(shuffled_order, np.arange(sonar_features.shape[0])))
print("Same multiset of rows (just reordered):",
      np.array_equal(np.sort(shuffled_sonar, axis=0), np.sort(sonar_features, axis=0)))


Shuffled array shares memory with original: False
Same shape: True
Row order actually changed: True
Same multiset of rows (just reordered): True


`np.shares_memory` confirms the shuffled array is a genuine copy (`False`),
consistent with Section 6's rule that fancy indexing always copies. The row
order is genuinely different from the original, but every row's actual values
are preserved (sorting both arrays along axis 0 gives the same result), so this
really is a shuffle, not a corruption -- exactly what you'd want before, e.g.,
a train/test split.
